In [4]:
import pandas as pd
import json
import re
import os
from pathlib import Path

In [43]:
DEFAULT_CONTROL_QUESTION = {
    "question": "What was the question about?",
    "correctAnswer": "Correct answer",
    "wrongAnswers": [
        "Wrong Answer",
        "Wrong Answer",
        "Wrong Answer",
        "Wrong Answer",
    ],
}
FEVER_LABEL_MAP = {
    "supported": True,
    "supports": True,
    "refuted": False,
    "refutes": False,
    "not enough evidence": None,
    "not enough info": None,
    "conflicting evidence/cherry-picking": None,
    "conflicting evidence": None,
    "cherry-picking": None,
}

FEVER_LABELS = [
    "Supported",
    "Refuted",
    "Not Enough Evidence",
    "Conflicting Evidence/Cherry-picking",
]

def capitalize_sentences(text):
    text = text.strip()
    if not text:
        return text
    text = text[0].upper() + text[1:]
    # capitalize next letter after whitespace
    text = re.sub(r"([.!?])([ \t]+)([a-z])", lambda m: m.group(1) + m.group(2) + m.group(3).upper(), text)
    # capitalize next letter after newlines
    text = re.sub(r"(\n+)([a-z])", lambda m: m.group(1) + m.group(2).upper(), text)
    return text

def parse_boolq_task(task):
    task = task.strip()

    q_match = re.search(r"Question:\s*(.*?)(?=\n\s*\n|\n\s*Passage:|\Z)", task, re.DOTALL | re.IGNORECASE)
    question_raw = q_match.group(1).strip() if q_match else ""
    question = capitalize_sentences(question_raw)
    if question and not question.endswith("?"):
        question += "?"

    p_match = re.search(r"Passage:\s*(.*?)(?=\n\s*\n\s*Answer the question|\n\s*Answer the question|\Z)",task, re.DOTALL | re.IGNORECASE)
    passage_raw = p_match.group(1).strip() if p_match else ""
    passage = capitalize_sentences(passage_raw)

    return question, passage

def parse_fever_task(task):
    task = task.strip()

    claim_match = re.search(r"Claim:\s*(.*?)(?=\n\s*\n|\n\s*Evidence:|\Z)", task, re.DOTALL | re.IGNORECASE)
    claim_raw = claim_match.group(1).strip() if claim_match else ""
    claim_raw = re.sub(r'^[\"\u201c\u2018]+|[\"\u201d\u2019]+$', "", claim_raw).strip()
    claim = capitalize_sentences(claim_raw)

    evidence_match = re.search(r"Evidence:\s*(.*?)(?=\n\s*\n\s*Choose\s+which|\n\s*Choose\s+which|\Z)", task, re.DOTALL | re.IGNORECASE)
    evidence_raw = evidence_match.group(1).strip() if evidence_match else ""
    evidence = capitalize_sentences(evidence_raw)

    labels_match = re.search(r"Choose\s+which\b.*?:\s*\n(.*?)(?=\Z)", task, re.DOTALL | re.IGNORECASE)
    if labels_match:
        raw_lines = [l.strip() for l in labels_match.group(1).splitlines() if l.strip()]
        label_options = raw_lines if raw_lines else FEVER_LABELS
    else:
        label_options = FEVER_LABELS

    return claim, evidence, label_options


def bold_to_sentiment(text):
    return re.sub(
        r"\*\*(.+?)\*\*",
        lambda m: f"<mark>{m.group(1).strip()}</mark>",
        text,
        flags=re.DOTALL,
    )

def fever_label_to_bool(label: str):
    return FEVER_LABEL_MAP.get(label.strip().lower(), None)

def parse_zebralogic_task(task):
    # CONTEXT
    ctx_m = re.search(r"CONTEXT:\s*(.*?)(?=\n\s*QUESTION:|\Z)", task, re.DOTALL | re.IGNORECASE)
    context_raw = ctx_m.group(1).strip() if ctx_m else task.strip()
    context_raw = re.sub(r"`([^`]+)`", r"\1", context_raw)   # strip backticks
    context = capitalize_sentences(context_raw)

    # QUESTION
    q_m = re.search(r"QUESTION:\s*(.*?)(?=\n\s*OPTIONS:|\Z)", task, re.DOTALL | re.IGNORECASE)
    question = capitalize_sentences(q_m.group(1).strip()) if q_m else ""

    # OPTIONS
    opt_m = re.search(r"OPTIONS:\s*(.*?)(?=\Z)", task, re.DOTALL | re.IGNORECASE)
    options = []
    if opt_m:
        for line in opt_m.group(1).splitlines():
            m = re.match(r"^[A-Z][).]\s*(.+)$", line.strip())
            if m:
                t = m.group(1).strip()
                options.append((t[0].upper() + t[1:]) if t else t)

    return context, question, options

def letter_to_option(letter, options):
    idx = ord(letter.upper()) - ord("A")
    return options[idx] if 0 <= idx < len(options) else letter

In [46]:
def build_dataset_items(items_df, explanations_df, dataset_name):
    subset = explanations_df[explanations_df["dataset"].str.lower() == dataset_name.lower()].drop(columns=['explainer_1', 'explainer_2']).copy()

    items = []
    for _idx, group in subset.groupby("original_index", sort=True):
        anchor = items_df[items_df['original_index']==_idx]
        label = anchor['label'].iloc[0]
        pred = anchor['predicted_label'].iloc[0]
        # Extract the three explanations per og_idx
        xai_map = {}
        for _, row in group.iterrows():
            xai_map[str(row["xai_type"]).lower().strip()] = row

        xai_features = {"truthfulness": None}

        # XAI feature content usually stored in explainer_3
        xai_features["highlightedContent"]         = bold_to_sentiment(xai_map["attribution"]["explainer_3"])
        task = xai_map["counterfactual"]["explainer_3"]
        p_match = re.search(
            r"Passage:\s*(.*?)(?=\s*Answer\s+the\s+question|\Z)",
            task, re.DOTALL | re.IGNORECASE
        )
        passage_raw = p_match.group(1).strip() if p_match else ""
        passage = capitalize_sentences(passage_raw)
        xai_features["counterfactualExplanation"]  = passage
        xai_features["naturalLanguageExplanation"] = xai_map["rationale"]["explainer_3"]

        # Shared fields
        base = {
            "id":              _idx,
            "isFalsePositive": False,
            "isTrueNegative":  False,
            "isQualification": False, 
            "category":        dataset_name,
            "xaiFeatures":     xai_features,
            "controlQuestion": DEFAULT_CONTROL_QUESTION,
        }
        # Dataset-specific fields
        task_text = xai_map["attribution"]["task"]
        if dataset_name.lower() == "zebralogic":
            context, question, options = parse_zebralogic_task(task_text)
            # print(f'{dataset_name.lower()}\n{options}\n{options[label]} == {options[pred]}')
            xai_features['truthfulness'] = options[label]
            item = {**base, "title": question, "content": context, "ratingType": "multiple-choice", "options": options}

        elif dataset_name.lower() == "boolq":
            question, passage = parse_boolq_task(task_text)
            # print(f'{dataset_name.lower()}\n{bool(label)} == {bool(pred)}')
            xai_features['truthfulness'] = bool(label)
            item = {**base, "title": question, "content": passage, "ratingType": "boolean"}
        
        elif dataset_name.lower() == "fever":
            claim, evidence, options = parse_fever_task(task_text)
            # print(f'{dataset_name.lower()}\n {options}\n{label} == {pred}')
            xai_features['truthfulness'] = options[label]
            item = {**base, "title": claim, "content": evidence, "ratingType": "multiple-choice", "options": options}
        items.append(item)

    return items


def generate_dataset_jsons(item_csv_path, ex_sv_path, output_dir = ".", datasets = ("boolq", "zebralogic", "fever")):
    os.makedirs(output_dir, exist_ok=True)
    explanations_df = pd.read_csv(ex_sv_path)
    items_df = pd.read_csv(item_csv_path)

    results = {}
    for name in datasets:
        items = build_dataset_items(items_df, explanations_df, name)
        out_path = os.path.join(output_dir, f"{name}-items.json")
        with open(out_path, "w", encoding="utf-8") as fh:
            json.dump(items, fh, indent=2, ensure_ascii=False)
        results[name] = items

    return results


In [47]:
results = generate_dataset_jsons(item_csv_path='dataset-items.csv', ex_sv_path="explanations.csv", output_dir="../src/data/")